# NYC Yellow Taxi Trip Data — Initial Exploration

This notebook performs an initial exploratory analysis of the January 2025
NYC Yellow Taxi trip dataset.

## Objectives

- Inspect the Parquet file metadata and schema.
- Load a manageable subset of the dataset.
- Review column names and data types.
- Identify missing values and duplicated records.
- Examine date ranges and categorical values.
- Calculate trip duration and average speed.
- Identify potentially invalid or unusual records.
- Create a reproducible sample for development and testing.

> This analysis initially uses only the first Parquet row group. Therefore,
> the results do not yet represent the complete month of January 2025.

## 1. Environment Setup

Import the libraries required for filesystem operations, numerical
calculations, tabular data analysis, and Parquet file inspection.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

### 1.1 Project Paths

Define the project root and the paths to the raw dataset and the development
sample.

Using `pathlib.Path` keeps the paths readable and independent of the operating
system's path separator.

In [2]:
PROJECT_ROOT = Path.cwd().parent

# Full Parquet dataset path
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    /"yellow_tripdata_2025-01.parquet"
)

# Development sample path
SAMPLE_PATH = (
    PROJECT_ROOT
    / "data"
    / "sample"
    / "yellow_tripdata_2025-01_sample.parquet"
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset exists: {DATA_PATH.exists()}")
print(
    f"Dataset path: "
    f"{DATA_PATH.relative_to(PROJECT_ROOT)}"
)

Project root: c:\Users\angel\Documents\Proyectos\urban-mobility-analytics
Dataset exists: True
Dataset path: data\raw\yellow_tripdata_2025-01.parquet


## 2. Parquet File Inspection

Inspect the Parquet metadata before loading the records into memory.

Parquet metadata provides information such as the total number of rows,
columns, and internal row groups without requiring the complete dataset to be
converted into a pandas DataFrame.

In [3]:
# Open the Parquet structure without loading all records into pandas.
parquet_file = pq.ParquetFile(DATA_PATH)

metadata_summary = {
    "rows": parquet_file.metadata.num_rows,
    "columns": parquet_file.metadata.num_columns,
    "row_groups": parquet_file.metadata.num_row_groups
}

metadata_summary

{'rows': 3475226, 'columns': 20, 'row_groups': 4}

### 2.1 Parquet Schema

Display the source columns and their Arrow data types.

This is the schema of the Parquet file. It should not be confused with a
PostgreSQL schema, which is used to organize database objects such as tables
and views.

In [4]:
print(parquet_file.schema_arrow)

VendorID: int32
tpep_pickup_datetime: timestamp[us]
tpep_dropoff_datetime: timestamp[us]
passenger_count: int64
trip_distance: double
RatecodeID: int64
store_and_fwd_flag: large_string
PULocationID: int32
DOLocationID: int32
payment_type: int64
fare_amount: double
extra: double
mta_tax: double
tip_amount: double
tolls_amount: double
improvement_surcharge: double
total_amount: double
congestion_surcharge: double
Airport_fee: double
cbd_congestion_fee: double


## 3. Load a Manageable Data Subset

Load only the first Parquet row group instead of the complete monthly dataset.

This provides enough records for initial exploration while reducing memory
usage and execution time.

In [5]:
# Row groups use zero-based indexing, so 0 refers to the first group.
first_row_group = parquet_file.read_row_group(0)

df = first_row_group.to_pandas()

print(f"Rows loaded: {len(df):,}")
print(f"Columns loaded: {len(df.columns)}")

Rows loaded: 1,048,576
Columns loaded: 20


### 3.1 DataFrame Dimensions

Confirm the number of rows and columns currently loaded into memory.

In [6]:
df.shape

(1048576, 20)

### 3.2 Initial Record Preview

Display the first records to understand the general structure and values of
the dataset.

In [7]:
df.head(10)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1,1.60,1,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1,0.50,1,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1,0.60,1,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3,0.52,1,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3,0.66,1,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0
5,2,2025-01-01 00:48:24,2025-01-01 01:08:26,2,2.63,1,N,239,68,2,19.1,1.0,0.5,0.00,0.0,1.0,24.10,2.5,0.0,0.0
6,1,2025-01-01 00:14:47,2025-01-01 00:16:15,0,0.40,1,N,170,170,1,4.4,3.5,0.5,2.35,0.0,1.0,11.75,2.5,0.0,0.0
7,1,2025-01-01 00:39:27,2025-01-01 00:51:51,0,1.60,1,N,234,148,1,12.1,3.5,0.5,2.00,0.0,1.0,19.10,2.5,0.0,0.0
8,1,2025-01-01 00:53:43,2025-01-01 01:13:23,0,2.80,1,N,148,170,1,19.1,3.5,0.5,3.00,0.0,1.0,27.10,2.5,0.0,0.0
9,2,2025-01-01 00:00:02,2025-01-01 00:09:36,1,1.71,1,N,237,262,2,11.4,1.0,0.5,0.00,0.0,1.0,16.40,2.5,0.0,0.0


## 4. Column Structure and Data Types

List the source columns in their original order before applying any
transformations.

In [8]:
for index, column in enumerate(df.columns, start=1):
    print(f"{index:02}. {column}")

01. VendorID
02. tpep_pickup_datetime
03. tpep_dropoff_datetime
04. passenger_count
05. trip_distance
06. RatecodeID
07. store_and_fwd_flag
08. PULocationID
09. DOLocationID
10. payment_type
11. fare_amount
12. extra
13. mta_tax
14. tip_amount
15. tolls_amount
16. improvement_surcharge
17. total_amount
18. congestion_surcharge
19. Airport_fee
20. cbd_congestion_fee


### 4.1 Data Types and Memory Usage

Inspect each column's pandas data type, non-null count, and approximate memory
usage.

The `deep` option provides a more accurate estimate for columns containing
Python objects or strings.

In [9]:
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 1048576 entries, 0 to 1048575
Data columns (total 20 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   VendorID               1048576 non-null  int32         
 1   tpep_pickup_datetime   1048576 non-null  datetime64[us]
 2   tpep_dropoff_datetime  1048576 non-null  datetime64[us]
 3   passenger_count        1048576 non-null  int64         
 4   trip_distance          1048576 non-null  float64       
 5   RatecodeID             1048576 non-null  int64         
 6   store_and_fwd_flag     1048576 non-null  str           
 7   PULocationID           1048576 non-null  int32         
 8   DOLocationID           1048576 non-null  int32         
 9   payment_type           1048576 non-null  int64         
 10  fare_amount            1048576 non-null  float64       
 11  extra                  1048576 non-null  float64       
 12  mta_tax                1048576 non-null

### 4.2 Data Type Summary

Create a compact table that maps each column to its current pandas data type.
This will later help define appropriate PostgreSQL column types.

In [10]:
column_types = pd.DataFrame(
    {
        "column": df.columns,
        "pandas_dtype": df.dtypes.astype(str).values,
    }
)

column_types

,column,pandas_dtype
0,VendorID,int32
1,tpep_pickup_datetime,datetime64[us]
2,tpep_dropoff_datetime,datetime64[us]
3,passenger_count,int64
4,trip_distance,float64
5,RatecodeID,int64
6,store_and_fwd_flag,str
7,PULocationID,int32
8,DOLocationID,int32
9,payment_type,int64


## 5. Descriptive Statistics

Calculate summary statistics for the numerical columns.

The transposed layout makes it easier to compare the count, mean, standard
deviation, quartiles, minimum, and maximum of each variable.

In [11]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
VendorID,1048576.0,1.782692,1.0,2.0,2.0,2.0,7.0,0.418474
tpep_pickup_datetime,1048576,2025-01-07 05:46:18.637088,2024-12-31 20:47:55,2025-01-04 11:07:44.750000,2025-01-07 13:38:23,2025-01-10 01:03:43,2025-01-12 23:19:37,NaN
tpep_dropoff_datetime,1048576,2025-01-07 06:01:16.467354,2024-12-31 20:54:00,2025-01-04 11:20:40.250000,2025-01-07 13:53:30,2025-01-10 01:18:16.250000,2025-01-13 17:08:02,NaN
passenger_count,1048576.0,1.356582,0.0,1.0,1.0,1.0,9.0,0.811195
trip_distance,1048576.0,3.29718,0.0,0.97,1.63,3.1,4020.04,6.449413
RatecodeID,1048576.0,2.394652,1.0,1.0,1.0,1.0,99.0,11.214913
PULocationID,1048576.0,165.717153,1.0,132.0,162.0,234.0,265.0,63.39548
DOLocationID,1048576.0,164.796133,1.0,114.0,162.0,234.0,265.0,69.488682
payment_type,1048576.0,1.251288,1.0,1.0,1.0,1.0,5.0,0.614727
fare_amount,1048576.0,17.695368,-900.0,8.6,12.1,19.8,900.0,19.70194


## 6. Missing Value Analysis

Count missing values in each column and calculate their percentage relative to
the number of loaded records.

In [12]:
null_summary = (
    df.isna()
    .sum(axis=0)
    .to_frame(name="null_count")
)

# Calculate the percentage of null values for each column.

null_summary["null_percentage"] = (
    null_summary["null_count"]
    / len(df) 
    * 100
)

null_summary = null_summary.sort_values(
    by="null_percentage",
    ascending=False,
)

null_summary

,null_count,null_percentage
VendorID,0,0.0
tpep_pickup_datetime,0,0.0
tpep_dropoff_datetime,0,0.0
passenger_count,0,0.0
trip_distance,0,0.0
RatecodeID,0,0.0
store_and_fwd_flag,0,0.0
PULocationID,0,0.0
DOLocationID,0,0.0
payment_type,0,0.0


### 6.1 Columns With the Highest Null Percentages

Display the first ten rows of the null summary after sorting it in descending
order.

In [13]:
null_summary.head(10)

,null_count,null_percentage
VendorID,0,0.0
tpep_pickup_datetime,0,0.0
tpep_dropoff_datetime,0,0.0
passenger_count,0,0.0
trip_distance,0,0.0
RatecodeID,0,0.0
store_and_fwd_flag,0,0.0
PULocationID,0,0.0
DOLocationID,0,0.0
payment_type,0,0.0


### 6.2 Columns Containing Missing Values

Filter the summary to verify whether any column contains at least one null
value.

In [14]:
null_summary[null_summary["null_count"] > 0]

,null_count,null_percentage


## 7. Complete Duplicate Analysis

Count records where every column has the same value as a previous row.

This only identifies fully identical records. It does not guarantee that the
dataset contains no duplicated trips, because the source does not provide a
unique trip identifier.

In [15]:
duplicate_count = df.duplicated().sum()
print(f"Completely duplicated rows: {duplicate_count:,}")

Completely duplicated rows: 0


## 8. Date Range Analysis

Inspect the earliest and latest pickup and drop-off timestamps in the loaded
row group.

Although the file represents January 2025, some records may fall outside the
expected monthly range.

In [16]:
date_summary = df[
    [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
    ]
].agg(["min", "max"])

date_summary

,tpep_pickup_datetime,tpep_dropoff_datetime
min,2024-12-31 20:47:55,2024-12-31 20:54:00
max,2025-01-12 23:19:37,2025-01-13 17:08:02


## 9. Derived Trip Metrics

### 9.1 Trip Duration

Calculate trip duration in minutes by subtracting the pickup timestamp from
the drop-off timestamp.

Non-positive values may indicate invalid or incorrectly recorded timestamps.

In [17]:
# Subtract the pickup timestamp from the drop-off timestamp and convert the
# resulting timedelta from seconds to minutes.
df["trip_duration_minutes"] = (
    df["tpep_dropoff_datetime"]
    - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

df["trip_duration_minutes"].describe()

count    1.048576e+06
mean     1.496384e+01
std      2.991252e+01
min     -5.603333e+01
25%      6.833333e+00
50%      1.123333e+01
75%      1.820000e+01
max      5.022133e+03
Name: trip_duration_minutes, dtype: float64

### 9.2 Average Trip Speed

Calculate average speed in miles per hour using trip distance and trip
duration.

Speed is calculated only when duration is greater than zero to avoid invalid
or infinite results.

In [18]:
# Only positive durations can produce a meaningful average speed.
valid_duration = df["trip_duration_minutes"] > 0

df["average_speed_mph"] = np.where(
    valid_duration,
    df["trip_distance"]
    / (df["trip_duration_minutes"] / 60),
    np.nan,
)

df["average_speed_mph"].describe()

count    1.048196e+06
mean     1.199714e+01
std      4.963566e+01
min      0.000000e+00
25%      7.370474e+00
50%      9.730258e+00
75%      1.342160e+01
max      1.656000e+04
Name: average_speed_mph, dtype: float64

## 10. Initial Data Quality Assessment

Define preliminary quality indicators for duplicated records, invalid
durations, unusual distances, negative monetary values, timestamps outside the
expected month, and unusually high average speeds.

These checks are exploratory. They do not yet represent the final validation
rules used by the data pipeline.

In [19]:
january_start = pd.Timestamp("2025-01-01")
february_start = pd.Timestamp("2025-02-01")

quality_summary = pd.Series(
    {
        "complete_duplicates": df.duplicated().sum(),
        "non_positive_duration": (
            df["trip_duration_minutes"] <= 0
        ).sum(),
        "negative_distance": (
            df["trip_distance"] < 0
        ).sum(),
        "zero_distance": (
            df["trip_distance"] == 0
        ).sum(),
        "negative_fare": (
            df["fare_amount"] < 0
        ).sum(),
        "negative_total_amount": (
            df["total_amount"] < 0
        ).sum(),
        "pickup_outside_january": (
            (df["tpep_pickup_datetime"] < january_start)
            | (df["tpep_pickup_datetime"] >= february_start)
        ).sum(),
        "speed_over_80_mph": (
            df["average_speed_mph"] > 80
        ).sum(),
    },
    name="record_count",
)

quality_summary.to_frame()

,record_count
complete_duplicates,0
non_positive_duration,380
negative_distance,0
zero_distance,14709
negative_fare,22836
negative_total_amount,22930
pickup_outside_january,21
speed_over_80_mph,244


### 10.1 Data Quality Percentages

Convert each quality issue count into a percentage of the loaded records and
sort the results from most frequent to least frequent.

In [20]:
quality_report = quality_summary.to_frame()

quality_report["percentage"] = (
    quality_report["record_count"] / len(df) * 100
)

quality_report = quality_report.sort_values(
    by="percentage",
    ascending=False,
)

quality_report

,record_count,percentage
negative_total_amount,22930,2.186775
negative_fare,22836,2.177811
zero_distance,14709,1.402760
non_positive_duration,380,0.036240
speed_over_80_mph,244,0.023270
pickup_outside_january,21,0.002003
negative_distance,0,0.000000
complete_duplicates,0,0.000000


## 11. Categorical Value Inspection

Inspect the frequency of important coded fields.

The NYC TLC dataset represents several categories using numerical or short
text codes. These codes will later be mapped to descriptive dimension tables.

### 11.1 Payment Type

Count each payment method code, including missing values if they exist.

In [21]:
df["payment_type"].value_counts(
    dropna=False
).sort_index()

payment_type
1    854503
2    154710
3      9305
4     30057
5         1
Name: count, dtype: int64

### 11.2 Technology Provider

Inspect the distribution of `VendorID`, which identifies the technology
provider that submitted the trip record.

In [22]:
df["VendorID"].value_counts(
    dropna=False
).sort_index()

VendorID
1    228744
2    819656
7       176
Name: count, dtype: int64

### 11.3 Rate Code

Inspect the rate code assigned to each trip. These values represent different
fare calculation rules and special trip categories.

In [23]:
df["RatecodeID"].value_counts(
    dropna=False
).sort_index()

RatecodeID
1     978424
2      38929
3       3497
4       2725
5      11078
6          5
99     13918
Name: count, dtype: int64

### 11.4 Passenger Count

Inspect passenger counts to identify common values, missing codes, zero
passenger records, or unusually high counts.

In [24]:
df["passenger_count"].value_counts(
    dropna=False,
).sort_index()

passenger_count
0      8701
1    796364
2    161382
3     40720
4     30228
5      6823
6      4350
7         2
8         4
9         2
Name: count, dtype: int64

### 11.5 Store-and-Forward Flag

Inspect whether trip records were temporarily stored in the vehicle before
being transmitted to the provider.

The expected values are normally short categorical codes such as `Y` and `N`.

In [25]:
df["store_and_fwd_flag"].value_counts(
    dropna=False,
).sort_index()

store_and_fwd_flag
N    1045819
Y       2757
Name: count, dtype: int64

## 12. Negative Transaction Analysis

Examine records with a negative total amount.

Negative monetary values should not be removed automatically because they may
represent refunds, reversals, corrections, or other legitimate administrative
transactions.

In [26]:
# Select the monetary and categorical fields needed to understand negative
# transactions without modifying the original DataFrame.
negative_transactions = df.loc[
    df["total_amount"] < 0,
    [
        "VendorID",
        "RatecodeID",
        "payment_type",
        "trip_distance",
        "fare_amount",
        "extra",
        "mta_tax",
        "tip_amount",
        "tolls_amount",
        "total_amount",
    ],
]

negative_transactions.head(20)

,VendorID,RatecodeID,payment_type,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,total_amount
17,2,1,2,0.71,-7.2,-1.0,-0.5,3.66,0.00,-8.54
22,2,1,4,0.69,-6.5,-1.0,-0.5,0.00,0.00,-11.50
104,2,1,4,0.97,-16.3,-1.0,-0.5,0.00,0.00,-21.30
149,2,1,2,1.42,-12.1,-1.0,-0.5,0.00,0.00,-17.10
202,2,1,4,0.60,-7.2,-1.0,-0.5,0.00,0.00,-12.20
212,2,1,4,1.88,-14.2,-1.0,-0.5,0.00,0.00,-19.20
364,2,1,4,0.01,-3.0,-1.0,-0.5,0.00,0.00,-5.50
400,2,1,2,0.60,-6.5,-1.0,-0.5,0.00,0.00,-11.50
492,2,1,2,3.84,-24.0,-1.0,-0.5,0.00,0.00,-29.00
640,2,1,2,0.92,-6.5,-1.0,-0.5,0.00,0.00,-11.50


### 12.1 Negative Transaction Summary

Calculate descriptive statistics for the monetary components of negative
transactions to determine which fields are being reversed or adjusted.

In [27]:
negative_transactions[
    [
        "fare_amount",
        "extra",
        "mta_tax",
        "tip_amount",
        "tolls_amount",
        "total_amount",
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
fare_amount,22930.0,-22.575067,29.286983,-900.00,-26.10,-12.80,-7.20,0.00
extra,22930.0,-0.940100,1.452986,-7.50,-1.00,0.00,0.00,0.00
mta_tax,22930.0,-0.477606,0.103422,-0.50,-0.50,-0.50,-0.50,0.00
tip_amount,22930.0,0.087823,1.578043,-86.00,0.00,0.00,0.00,123.34
tolls_amount,22930.0,-0.707976,2.905629,-126.94,0.00,0.00,0.00,0.00
total_amount,22930.0,-28.224548,30.691070,-901.00,-31.45,-17.55,-12.25,-0.55


## 13. Potentially Invalid or Unusual Records

### 13.1 Zero-Distance Trips

Inspect trips with zero recorded distance to determine whether they contain
valid durations, charges, or different pickup and drop-off locations.

In [28]:
zero_distance_trips = df.loc[
    df["trip_distance"] == 0,
    [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_duration_minutes",
        "PULocationID",
        "DOLocationID",
        "fare_amount",
        "total_amount",
        "RatecodeID",
    ],
]

zero_distance_trips.head(20)

,tpep_pickup_datetime,tpep_dropoff_datetime,trip_duration_minutes,PULocationID,DOLocationID,fare_amount,total_amount,RatecodeID
92,2025-01-01 00:49:48,2025-01-01 00:49:48,0.000000,87,264,20.06,20.06,1
204,2025-01-01 00:37:43,2025-01-01 00:37:53,0.166667,148,148,12.00,17.50,5
358,2025-01-01 00:57:08,2025-01-01 00:57:16,0.133333,141,141,30.00,33.50,5
505,2025-01-01 00:27:40,2025-01-01 00:59:30,31.833333,168,76,50.50,58.94,1
619,2025-01-01 00:56:49,2025-01-01 00:56:54,0.083333,164,164,20.00,30.55,5
706,2025-01-01 00:42:42,2025-01-01 00:42:44,0.033333,261,261,3.00,10.35,1
1073,2025-01-01 00:43:22,2025-01-01 00:46:18,2.933333,48,48,5.00,9.00,5
1345,2025-01-01 00:24:26,2025-01-01 00:24:34,0.133333,60,60,20.00,25.20,5
1354,2025-01-01 00:14:57,2025-01-01 00:17:44,2.783333,79,79,3.00,8.00,1
1496,2025-01-01 00:49:10,2025-01-01 00:49:14,0.066667,143,143,85.00,115.05,5


### 13.2 Unusually High Average Speeds

Inspect records whose calculated average speed exceeds 80 mph.

This threshold is used only as an exploratory indicator and should not yet be
treated as a definitive rejection rule.

In [29]:
high_speed_trips = df.loc[
    df["average_speed_mph"] > 80,
    [
        "trip_distance",
        "trip_duration_minutes",
        "average_speed_mph",
        "PULocationID",
        "DOLocationID",
        "fare_amount",
        "total_amount",
    ],
].sort_values(
    by="average_speed_mph",
    ascending=False,
)

high_speed_trips.head(20)

,trip_distance,trip_duration_minutes,average_speed_mph,PULocationID,DOLocationID,fare_amount,total_amount
771923,13.80,0.050000,16560.000000,216,216,3.00,8.75
727110,17.60,0.066667,15840.000000,100,100,70.00,74.75
402091,12.50,0.050000,15000.000000,249,249,70.00,74.75
484198,18.52,0.083333,13334.400000,132,132,70.00,76.50
830569,10.80,0.050000,12960.000000,41,41,3.00,4.50
282864,3.50,0.016667,12600.000000,100,100,23.30,27.30
451343,16.80,0.083333,12096.000000,63,63,3.00,4.50
631599,16.90,0.100000,10140.000000,237,237,3.00,7.00
145338,2.61,0.016667,9396.000000,10,10,12.10,14.60
875888,5.10,0.033333,9180.000000,87,87,3.00,8.75


### 13.3 Non-Positive Trip Durations

Inspect trips where the drop-off time is equal to or earlier than the pickup
time.

In [30]:
invalid_duration_trips = df.loc[
    df["trip_duration_minutes"] <= 0,
    [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_duration_minutes",
        "trip_distance",
        "fare_amount",
        "total_amount",
    ],
]

invalid_duration_trips.head(20)

,tpep_pickup_datetime,tpep_dropoff_datetime,trip_duration_minutes,trip_distance,fare_amount,total_amount
92,2025-01-01 00:49:48,2025-01-01 00:49:48,0.0,0.00,20.06,20.06
11604,2025-01-01 01:42:36,2025-01-01 01:42:36,0.0,0.00,3.00,8.00
13820,2025-01-01 02:13:25,2025-01-01 02:13:25,0.0,0.00,114.00,114.00
14605,2025-01-01 02:09:52,2025-01-01 02:09:52,0.0,0.00,3.00,8.00
14607,2025-01-01 02:49:40,2025-01-01 02:49:40,0.0,0.00,3.00,8.00
16093,2025-01-01 02:21:20,2025-01-01 02:21:20,0.0,0.00,16.30,21.30
19599,2025-01-01 03:46:16,2025-01-01 03:46:16,0.0,0.00,31.91,31.91
31891,2025-01-01 12:44:07,2025-01-01 12:44:07,0.0,1.56,17.00,31.00
34038,2025-01-01 12:21:49,2025-01-01 12:21:49,0.0,0.00,14.20,18.20
34951,2025-01-01 12:53:45,2025-01-01 12:53:45,0.0,0.00,21.55,21.55


## 14. Create a Reproducible Development Sample

Create a small random sample that can be stored in the repository and used for
local development, automated tests, and pipeline experimentation.

A fixed random seed ensures that the same source DataFrame produces the same
sample every time.

In [31]:
sample_size = min(5_000, len(df))

sample_df = df.sample(
    n=sample_size,
    random_state=42,
)

SAMPLE_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

sample_df.to_parquet(
    SAMPLE_PATH,
    index=False,
)

print(f"Sample rows: {len(sample_df):,}")
print(
    f"Sample saved to: "
    f"{SAMPLE_PATH.relative_to(PROJECT_ROOT)}"
)
print(
    f"Sample size: "
    f"{SAMPLE_PATH.stat().st_size / 1024**2:.2f} MB"
)

Sample rows: 5,000
Sample saved to: data\sample\yellow_tripdata_2025-01_sample.parquet
Sample size: 0.22 MB


## 15. Preliminary Findings

The first Parquet row group contains 1,048,576 records. The initial analysis
identified the following points:

- No complete duplicate rows were found.
- No negative trip distances were found.
- A small number of pickup timestamps fall outside January 2025.
- Approximately 2.2% of records contain negative fare or total amounts.
- Approximately 1.4% of records have zero recorded trip distance.
- A small number of records have non-positive trip duration.
- Some calculated average speeds exceed 80 mph.
- Negative monetary records require further investigation before they can be
  classified as invalid.
- Zero-distance and high-speed records should be flagged rather than removed
  automatically.

## Current Limitations

- Only the first Parquet row group was analyzed.
- The current results do not represent the complete month.
- The initial development sample was created from the first row group only.
- The dataset does not contain a direct unique trip identifier.
- The current quality thresholds are exploratory and may change after reviewing
  the official data dictionary and the complete dataset.

## Next Steps

- Review the official definitions of categorical codes.
- Build a sample that includes records from every row group.
- Define formal data quality rules.
- Create the raw PostgreSQL tables.
- Develop reusable Python transformation and validation functions.